# 06. Explainable AI: Grad-CAM Saliency Maps for Sudoku Digit Recognition

### Overview
Deep neural networks are often treated as black boxes. 
**Gradient-weighted Class Activation Mapping (Grad-CAM)** provides visual explainability by computing the gradients of the target class score with respect to the feature activation maps of the final convolutional layer.

### What Grad-CAM Reveals:
- Identifies which structural strokes of each digit (e.g. the loop of a `6`, the crossbar of a `4`, the diagonal of a `7`) drove the classification.
- Validates that the network is focusing on the centered digit rather than edge artifacts or lighting noise.

In [ ]:
import os
import sys
from pathlib import Path

def find_root() -> Path:
    p = Path.cwd()
    for parent in [p] + list(p.parents):
        if (parent / "models" / "digit_classifier.keras").exists():
            return parent
    return Path(os.path.abspath("..")).resolve()

PROJECT_ROOT = find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from backend.cv_pipeline import find_grid_corners, warp_grid, split_cells

plt.rcParams['figure.dpi'] = 110

# Load trained model
model_path = PROJECT_ROOT / "models" / "digit_classifier.keras"
model = tf.keras.models.load_model(str(model_path))

class_names = ["1", "2", "3", "4", "5", "6", "7", "8", "9", "empty"]
print(f"Loaded CNN Model: {model_path.name}")

## 1. Locating the Target Convolutional Layer
We dynamically locate the final `Conv2D` layer in the sequential architecture.

In [ ]:
def find_last_conv_layer(model: tf.keras.Model) -> str:
    """Finds the name of the last Conv2D layer in the model."""
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            return layer.name
    raise ValueError("No Conv2D layer found in model.")

last_conv_name = find_last_conv_layer(model)
print("Target Convolutional Layer:", last_conv_name)

## 2. Grad-CAM Heatmap Calculation
Using `tf.GradientTape`, we record the forward pass activations of the last convolutional layer and backpropagate the target class logit to calculate feature importance weights.

In [ ]:
def make_gradcam_heatmap(
    img_tensor: tf.Tensor,
    model: tf.keras.Model,
    last_conv_layer_name: str,
    pred_index: int = None
) -> np.ndarray:
    """Generates a 2D Grad-CAM heatmap for a single image tensor."""
    grad_model = tf.keras.models.Model(
        inputs=[model.inputs],
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_tensor)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy()

## 3. Heatmap Superimposition
Overlay the colormapped heatmap onto the input cell patch.

In [ ]:
def overlay_gradcam(cell_img: np.ndarray, heatmap: np.ndarray, alpha: float = 0.5) -> np.ndarray:
    """Superimpose the heatmap on the grayscale cell image."""
    resized_heatmap = cv2.resize(heatmap, (cell_img.shape[1], cell_img.shape[0]))
    colored_heatmap = cv2.applyColorMap(np.uint8(255 * resized_heatmap), cv2.COLORMAP_JET)
    
    cell_bgr = cv2.cvtColor(cell_img, cv2.COLOR_GRAY2BGR)
    overlay = cv2.addWeighted(cell_bgr, 1 - alpha, colored_heatmap, alpha, 0)
    return cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)

## 4. Grad-CAM Analysis on Sample Sudoku Cells
Extract cells from a sample image, predict digits, and visualize Grad-CAM saliency.

In [ ]:
sample_path = PROJECT_ROOT / "data" / "raw" / "sample2.png"
if not sample_path.exists():
    sample_path = PROJECT_ROOT / "data" / "test_set" / "sample1.png"

img = cv2.imread(str(sample_path))
corners = find_grid_corners(img)
warped, _ = warp_grid(img, corners)
warped_gray = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)
cells = split_cells(warped_gray)

# Pick first 6 non-empty cells
selected_cells = []
for cell in cells:
    _, thresh = cv2.threshold(cell, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    resized = cv2.resize(thresh, (28, 28))
    inp = (resized.astype("float32") / 255.0).reshape(1, 28, 28, 1)
    pred = model.predict(inp, verbose=0)
    cls_idx = np.argmax(pred)
    label = class_names[cls_idx]
    if label != "empty":
        selected_cells.append((cell, inp, label, np.max(pred)))
        if len(selected_cells) >= 6:
            break

fig, axes = plt.subplots(len(selected_cells), 3, figsize=(9, len(selected_cells) * 2.5))
for i, (cell, inp, label, conf) in enumerate(selected_cells):
    heatmap = make_gradcam_heatmap(inp, model, last_conv_name)
    overlay = overlay_gradcam(cell, heatmap)
    
    axes[i, 0].imshow(cell, cmap="gray")
    axes[i, 0].set_title(f"Cell {i+1}")
    axes[i, 0].axis("off")
    
    axes[i, 1].imshow(heatmap, cmap="jet")
    axes[i, 1].set_title(f"Grad-CAM Heatmap")
    axes[i, 1].axis("off")
    
    axes[i, 2].imshow(overlay)
    axes[i, 2].set_title(f"Predicted: '{label}' ({conf*100:.1f}%)")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()